# Bronchoscopy sim-to-real smoke run

This notebook verifies the repository setup on a connected Colab runtime before using real airway data. It does not claim photorealistic research quality.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/broncho-sim2real')
if not repo.exists():
    subprocess.run([
        'git', 'clone', '-q',
        'https://github.com/stefanasandei/broncho-sim2real.git',
        str(repo),
    ], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt')], check=True)
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / 'src'))

import torch
print('Repository:', repo)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import sys
import torch

sys.path.insert(0, '/content/broncho-sim2real')
sys.path.insert(0, '/content/broncho-sim2real/src')
from src.smoke_reproduction import train_smoke

checkpoint = train_smoke(
    Path('/content/broncho-sim2real/runs/colab-smoke'),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    size=32,
    batches=100,
    steps=20,
    seed=7,
)
print('Saved:', checkpoint)
assert checkpoint.exists()

In [ ]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import torch

repo = Path('/content/broncho-sim2real')
depth_path = repo / 'outputs/colab-smoke-depth.npy'
depth_path.parent.mkdir(parents=True, exist_ok=True)
np.save(depth_path, np.full((32, 32), 0.5, dtype=np.float32))
output_dir = repo / 'outputs/colab-smoke'
subprocess.run([
    sys.executable, 'src/sample.py', str(depth_path),
    '--checkpoint', str(repo / 'runs/colab-smoke/checkpoints/smoke.pt'),
    '--output', str(output_dir), '--samples', '4', '--steps', '20',
    '--image-size', '32', '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
], cwd=repo, check=True)
print('Generated files:', sorted(path.name for path in output_dir.glob('*.png')))